# Windows、macOS与Linux开发差异

## 它们分别是什么

| 系统 | 常见位置 | 开发语境里的特点 |
| --- | --- | --- |
| Windows | 个人电脑、企业办公环境 | 图形界面强，路径和 Shell 与 Unix 系不同，常配合 WSL 开发 |
| macOS | 个人电脑、移动端开发、前端开发 | 类 Unix 环境，终端体验接近 Linux，常见于 iOS / macOS 开发 |
| Linux | 服务器、容器、CI/CD、云环境 | 线上部署最常见，命令行和权限模型是工程基础 |

## 路径差异

路径是最常见的跨系统坑。

| 差异 | Windows | macOS / Linux |
| --- | --- | --- |
| 路径分隔符 | `\` | `/` |
| 盘符 | `C:\Users\name` | 无盘符，常见 `/Users/name`、`/home/name` |
| 用户目录 | `C:\Users\name` | macOS: `/Users/name`，Linux: `/home/name` |

## 文件名大小写

Windows 和很多默认 macOS 文件系统对大小写不敏感，Linux 通常大小写敏感。

这意味着：

| 文件名 | 在 Windows / 常见 macOS | 在 Linux |
| --- | --- | --- |
| `User.ts` 和 `user.ts` | 可能被当成同一个文件 | 是两个不同文件 |
| 引入 `./User`，实际文件叫 `./user` | 可能本地正常 | 可能部署失败 |

这类问题在前端项目里很常见：本地开发没问题，CI 或 Linux 部署时报“找不到模块”。

写项目约定时，可以要求：

- 文件命名风格统一
- import 路径大小写必须和真实文件一致
- CI 在 Linux 环境跑一次构建或测试

## 换行符差异

不同系统历史上使用不同换行符。

| 系统 | 常见换行符 |
| --- | --- |
| Windows | CRLF |
| macOS / Linux | LF |

大多数现代编辑器能自动处理，但在脚本、Git diff、格式化工具里仍然可能出现问题。

常见表现：

- 明明只改了一行，Git 显示整个文件都变了
- Shell 脚本在 Linux 上执行失败
- 团队成员保存文件后格式来回变化

项目里通常用 `.gitattributes`、编辑器配置或格式化工具统一换行符。

## Shell 和命令差异

不同系统默认 Shell 不一样。

| 系统 | 常见 Shell / 命令环境 |
| --- | --- |
| Windows | PowerShell、CMD、Git Bash、WSL |
| macOS | zsh、bash |
| Linux | bash、zsh、sh |

## 工具安装方式差异

| 系统 | 常见安装方式 |
| --- | --- |
| Windows | 安装包、Microsoft Store、winget、scoop、chocolatey |
| macOS | 安装包、Homebrew |
| Linux | apt、dnf、yum、pacman、发行版包管理器 |

但项目内部依赖应该尽量交给项目自己的包管理器，例如：

- Python 项目用 uv、pip、poetry 管 Python 依赖
- Node 项目用 npm、pnpm、yarn 管前端依赖
- Rust 项目用 cargo 管 Rust 依赖

系统包管理器负责安装系统级工具，项目包管理器负责安装项目依赖。

## WSL 是什么位置

**WSL** 是 Windows Subsystem for Linux，让 Windows 用户可以在本机运行 Linux 环境。

它常用于：

- 在 Windows 上获得接近 Linux 的命令行体验
- 运行更接近服务器环境的开发工具
- 减少 Windows 与 Linux 部署环境之间的差异

## 本地、CI、服务器通常不一样

真实项目里经常会出现三套环境：

| 环境 | 常见系统 | 作用 |
| --- | --- | --- |
| 本地开发 | Windows / macOS / Linux | 写代码、调试、运行项目 |
| CI（持续集成） | 常见 Linux | 自动测试、构建、检查 |
| 服务器 / 容器 | 常见 Linux | 真实部署运行 |


- Docker 可以减少差异，但不能消灭所有差异
- 很多团队会把 CI 放在 Linux 上跑，就是为了尽早发现部署环境相关的问题。

## 在规格里怎么写

涉及开发、部署、脚本、文件上传、路径、自动化，就应该写清环境假设。

| 项目 | 规格里应该说明什么 |
| --- | --- |
| 本地开发 | 支持哪些系统，是否推荐 WSL |
| 运行环境 | 最终部署到 Linux、容器、云平台还是桌面端 |
| 脚本命令 | 目标 Shell 是 PowerShell 还是 bash / sh |
| 文件路径 | 使用项目相对路径，不写死个人电脑路径 |
| 权限 | 哪些目录需要读写，服务用什么权限运行 |
| CI 环境 | 是否要求在 Linux 上自动测试和构建 |

可以这样写：

- 本项目支持 Windows 和 macOS 本地开发，推荐 Windows 用户使用 WSL
- CI 和部署环境使用 Linux
- 文档中的部署脚本默认使用 bash
- 项目内文件路径必须使用相对路径，不允许写死本机绝对路径
- 上传目录和日志目录需要在部署时确认写入权限

## 面试常见问法

### 为什么项目在 Windows 能跑，部署到 Linux 失败？

常见原因包括路径分隔符不同、文件名大小写不一致、换行符问题、Shell 命令不兼容、权限不同、依赖版本或系统库不同。

### 为什么 CI 常常跑在 Linux 上？

因为服务器、容器和云部署环境通常以 Linux 为主。CI 在 Linux 上跑，可以提前发现大小写、权限、脚本和依赖相关问题。

## 常见坑

- **只在本地验证**：没有在接近部署环境的 CI 或容器里跑过。